# Llama 3.2-1B Activation-Based Estimation

Notebook này đã được viết lại theo repo sau refactor, dùng namespace canonical `carve_lm.llm`.

Nó chạy được trên Google Colab, Kaggle hoặc local nếu bạn mở notebook ngay trong repo.

Flow chính:
- bootstrap repo và dependencies
- load `meta-llama/Llama-3.2-1B`
- tạo calibration dataloader
- chạy activation-based estimation cho attention heads, attention groups, MLP neurons và embedding channels
- lưu score ra `artifacts/activation/llama32_1b_activation_scores.pt`
- evaluate baseline hoặc model đã prune bằng `lm-eval`


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/fannam/CarveLM.git"
REPO_NAME = "CarveLM"


def detect_runtime() -> str:
    try:
        import google.colab  # type: ignore
        return "colab"
    except ImportError:
        pass

    if Path("/kaggle/working").exists():
        return "kaggle"

    return "local"


def find_repo_root(start: Path) -> Path | None:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "carve_lm").exists():
            return candidate
    return None


runtime = detect_runtime()
repo_root = find_repo_root(Path.cwd())

if repo_root is None:
    if runtime == "colab":
        base_dir = Path("/content")
    elif runtime == "kaggle":
        base_dir = Path("/kaggle/working")
    else:
        base_dir = Path.cwd()

    repo_root = base_dir / REPO_NAME
    if not repo_root.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)

os.chdir(repo_root)

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[train,notebooks]"],
    cwd=repo_root,
    check=True,
)

print({"runtime": runtime, "repo_root": str(repo_root)})


In [ ]:
hf_token = (
    os.environ.get("HF_TOKEN")
    or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    or os.environ.get("HUGGINGFACE_TOKEN")
)

print("HF token found in environment:", bool(hf_token))
print("Nếu chưa có token hoặc model báo 401/403, chạy cell login bên dưới.")


In [ ]:
from huggingface_hub import notebook_login

if hf_token:
    print("HF token already found in environment. Skip interactive login.")
else:
    notebook_login()


In [ ]:
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorWithPadding

from carve_lm.llm.core import calculate_embedding_channels_global_score
from carve_lm.llm.estimators import available_estimators, create_estimator
from carve_lm.llm.pruners import create_pruner

os.environ["TOKENIZERS_PARALLELISM"] = "false"
auth_kwargs = {"token": hf_token} if hf_token else {}

MODEL_NAME = "meta-llama/Llama-3.2-1B"
DATASET_NAME = "wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"
SPLIT = "validation"
MAX_SAMPLES = 64
MAX_LENGTH = 256
BATCH_SIZE = 4
AGGREGATION = "l2"
SEED = 13

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else (
    torch.float16 if device == "cuda" else torch.float32
)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True

print(
    {
        "device": device,
        "dtype": str(dtype),
        "model": MODEL_NAME,
        "max_samples": MAX_SAMPLES,
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
    }
)

element_estimators = tuple(
    name for name in available_estimators() if name.endswith(".element")
)
print("Available element estimators:", element_estimators)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, **auth_kwargs)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
    **auth_kwargs,
)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

print(model.__class__.__name__)
print(
    {
        "hidden_size": model.config.hidden_size,
        "num_layers": model.config.num_hidden_layers,
        "num_heads": model.config.num_attention_heads,
        "num_key_value_heads": getattr(model.config, "num_key_value_heads", model.config.num_attention_heads),
        "intermediate_size": model.config.intermediate_size,
    }
)


## Build calibration dataloader

Estimator activation của repo mới vẫn nhận `dataloader` ở từng hàm `estimate_*`. Ở đây dùng một calibration set nhỏ từ WikiText-2 để giảm thời gian chạy trên Colab/Kaggle.


In [ ]:
raw_dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split=SPLIT)
text_column = "text"

raw_dataset = raw_dataset.filter(
    lambda example: example[text_column] is not None and example[text_column].strip() != ""
)

sample_count = min(MAX_SAMPLES, len(raw_dataset))
calibration_dataset = raw_dataset.shuffle(seed=SEED).select(range(sample_count))


def tokenize_batch(batch):
    return tokenizer(batch[text_column], truncation=True, max_length=MAX_LENGTH)


tokenized_dataset = calibration_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=calibration_dataset.column_names,
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8 if device == "cuda" else None,
    return_tensors="pt",
)

calibration_loader = DataLoader(
    tokenized_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
)

first_batch = next(iter(calibration_loader))
print(f"Calibration samples: {sample_count}")
{key: tuple(value.shape) for key, value in first_batch.items()}


In [ ]:
def topk_summary(score_dict, topk=5):
    summary = {}
    for key, scores in score_dict.items():
        values, indices = torch.topk(scores, k=min(topk, scores.numel()))
        summary[key] = [
            {"index": int(index), "score": float(value)}
            for value, index in zip(values.tolist(), indices.tolist())
        ]
    return summary


def preview_mapping(mapping, limit=3):
    return dict(list(mapping.items())[:limit])


In [ ]:
estimator = create_estimator("activation.element", model, device=device)
pruner = create_pruner("width", model, device=device)

print(type(estimator).__name__)
print(type(pruner).__name__)
print("Available width strategies:", pruner.available_strategies())


## Estimate attention head importance


In [ ]:
head_importance = estimator.estimate_attention_heads(calibration_loader, agg=AGGREGATION)
print(len(head_importance), head_importance[0].shape)


In [ ]:
head_topk = topk_summary(head_importance, topk=5)
preview_mapping(head_topk, limit=4)


## Estimate attention group importance

Repo mới có `estimate_attention_groups()` để score theo KV group trực tiếp. Với GQA/MQA, nên dùng score này nếu bạn prune theo group thay vì suy ra từ head score như flow cũ.


In [ ]:
group_importance = estimator.estimate_attention_groups(calibration_loader, agg=AGGREGATION)
print(len(group_importance), group_importance[0].shape)


In [ ]:
group_topk = topk_summary(group_importance, topk=5)
preview_mapping(group_topk, limit=4)


## Estimate MLP neuron importance


In [ ]:
neuron_importance = estimator.estimate_mlp_neurons(calibration_loader, agg=AGGREGATION)
print(len(neuron_importance), neuron_importance[0].shape)


In [ ]:
neuron_topk = topk_summary(neuron_importance, topk=5)
preview_mapping(neuron_topk, limit=4)


## Estimate embedding channel importance


In [ ]:
embedding_importance = estimator.estimate_embedding_channels(calibration_loader, agg=AGGREGATION)
print(len(embedding_importance), next(iter(embedding_importance.values())).shape)


In [ ]:
embedding_global_score = calculate_embedding_channels_global_score(embedding_importance)
embedding_values, embedding_indices = torch.topk(
    embedding_global_score,
    k=min(20, embedding_global_score.numel()),
)
embedding_topk = [
    {"channel": int(index), "score": float(value)}
    for value, index in zip(embedding_values.tolist(), embedding_indices.tolist())
]

print("Per-hook tensors:", list(embedding_importance.keys())[:5], "...")
embedding_topk[:10]


In [ ]:
output_path = repo_root / "artifacts" / "activation" / "llama32_1b_activation_scores.pt"
output_path.parent.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "model_name": MODEL_NAME,
        "aggregation": AGGREGATION,
        "dataset_name": DATASET_NAME,
        "dataset_config": DATASET_CONFIG,
        "split": SPLIT,
        "max_samples": MAX_SAMPLES,
        "max_length": MAX_LENGTH,
        "head_importance": head_importance,
        "group_importance": group_importance,
        "neuron_importance": neuron_importance,
        "embedding_importance": embedding_importance,
        "embedding_global_score": embedding_global_score,
    },
    output_path,
)

output_path


## Optional: prune attention groups with the new API

Cell này minh họa flow prune theo repo mới. `prune_attention_group()` bây giờ nhận `group_importance` trực tiếp, đây là cách nên dùng cho Llama 3.2-1B.


In [ ]:
original_groups = getattr(model.config, "num_key_value_heads", model.config.num_attention_heads)
target_group = max(1, original_groups // 2)

if target_group < original_groups:
    pruned_attention_group_model = pruner.prune_attention_group(
        group_importance=group_importance,
        target_group=target_group,
    )
    pruned_model = pruned_attention_group_model
    print(
        {
            "original_num_attention_heads": model.config.num_attention_heads,
            "original_num_key_value_heads": original_groups,
            "pruned_num_attention_heads": pruned_attention_group_model.config.num_attention_heads,
            "pruned_num_key_value_heads": pruned_attention_group_model.config.num_key_value_heads,
        }
    )
else:
    print("This model has no removable attention group under the chosen rule.")


## Optional: prune MLP neurons

Repo hiện tại cũng hỗ trợ prune MLP theo `neuron_importance` bằng `prune_mlp()`.


In [ ]:
original_intermediate_size = model.config.intermediate_size
target_num_neurons = max(1, original_intermediate_size // 2)

if target_num_neurons < original_intermediate_size:
    pruned_mlp_model = pruner.prune_mlp(
        neuron_importance=neuron_importance,
        target_num_neurons=target_num_neurons,
    )
    print(
        {
            "original_intermediate_size": original_intermediate_size,
            "pruned_intermediate_size": pruned_mlp_model.config.intermediate_size,
        }
    )
else:
    print("This model has no removable MLP neurons under the chosen rule.")


## Optional: prune embedding channels

Embedding-channel pruning dùng trực tiếp `embedding_importance` hoặc `embedding_global_score`. Ở đây dùng `embedding_importance` để pruner tự tính global score nhất quán với repo.


In [ ]:
original_hidden_size = model.config.hidden_size
target_embedding_size = max(1, original_hidden_size // 2)

if target_embedding_size < original_hidden_size:
    pruned_embedding_model = pruner.prune_embeddings(
        embedding_importance=embedding_importance,
        target_embedding_size=target_embedding_size,
    )
    print(
        {
            "original_hidden_size": original_hidden_size,
            "pruned_hidden_size": pruned_embedding_model.config.hidden_size,
            "head_dim": getattr(pruned_embedding_model.config, "head_dim", None),
        }
    )
else:
    print("This model has no removable embedding channels under the chosen rule.")


Nếu bị out-of-memory trên Colab/Kaggle, giảm `MAX_SAMPLES`, `MAX_LENGTH` hoặc `BATCH_SIZE` rồi chạy lại từ cell load dataset trở xuống.


## Evaluate with `lm-eval`

Phần này dùng Python API chính thức của `lm-eval` (`simple_evaluate` + `HFLM`) để evaluate trực tiếp model đang có trong memory, tránh phải load lại thêm một bản model nữa.

Lần đầu chạy trên Colab/Kaggle, `lm-eval` có thể tải thêm task metadata và dataset tương ứng.

Cell cuối sẽ tự evaluate mọi biến thể nào đang có trong notebook: baseline, attention-group pruned, MLP-pruned, embedding-pruned.

Để test flow nhanh, cứ giữ `LM_EVAL_LIMIT` nhỏ trước; khi đã ổn thì đặt về `None` để chạy full benchmark.


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "lm_eval[hf]>=0.4.0"],
    check=True,
)


In [ ]:
import json
from numbers import Number

import lm_eval
from lm_eval.models.huggingface import HFLM

LM_EVAL_TASKS = ["piqa", "hellaswag", "arc_easy"]
LM_EVAL_LIMIT = None  # Đặt về None nếu muốn chạy full benchmark.
LM_EVAL_NUM_FEWSHOT = 0
LM_EVAL_BATCH_SIZE = BATCH_SIZE if device == "cuda" else 1
LM_EVAL_MAX_LENGTH = None
FREE_BASELINE_FROM_GPU_BEFORE_PRUNED_EVAL = True

lm_eval_output_dir = repo_root / "artifacts" / "lm_eval"
lm_eval_output_dir.mkdir(parents=True, exist_ok=True)

print(
    {
        "tasks": LM_EVAL_TASKS,
        "limit": LM_EVAL_LIMIT,
        "num_fewshot": LM_EVAL_NUM_FEWSHOT,
        "batch_size": LM_EVAL_BATCH_SIZE,
        "max_length": LM_EVAL_MAX_LENGTH,
        "output_dir": str(lm_eval_output_dir),
    }
)


In [ ]:
def _to_jsonable(value):
    if value is None or isinstance(value, (str, bool, Number)):
        return value
    if isinstance(value, dict):
        return {str(key): _to_jsonable(val) for key, val in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [_to_jsonable(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (torch.dtype, torch.device)):
        return str(value)
    if hasattr(value, "tolist") and callable(value.tolist):
        try:
            return _to_jsonable(value.tolist())
        except Exception:
            pass
    if hasattr(value, "item") and callable(value.item):
        try:
            return _to_jsonable(value.item())
        except Exception:
            pass
    if hasattr(value, "__dict__"):
        try:
            return {
                str(key): _to_jsonable(val)
                for key, val in vars(value).items()
                if not str(key).startswith("_")
            }
        except Exception:
            pass
    return str(value)


def run_lm_eval_with_loaded_model(model_obj, tokenizer_obj, run_name, tasks=None):
    model_obj = model_obj.to(device)
    model_obj.eval()
    if device == "cuda":
        torch.cuda.empty_cache()

    lm = HFLM(
        pretrained=model_obj,
        tokenizer=tokenizer_obj,
        batch_size=LM_EVAL_BATCH_SIZE,
        max_length=LM_EVAL_MAX_LENGTH,
        device=device,
    )

    results = lm_eval.simple_evaluate(
        model=lm,
        tasks=tasks or LM_EVAL_TASKS,
        num_fewshot=LM_EVAL_NUM_FEWSHOT,
        limit=LM_EVAL_LIMIT,
        log_samples=False,
    )

    result_path = lm_eval_output_dir / f"{run_name}.json"
    try:
        result_path.write_text(
            json.dumps(_to_jsonable(results), indent=2, ensure_ascii=False, default=str)
        )
        print(f"Saved lm-eval results to: {result_path}")
    except Exception as exc:
        print(f"Warning: failed to save lm-eval results to {result_path}: {exc}")
    return results, result_path


def summarize_lm_eval(results, run_name):
    rows = []
    for task_name, metrics in results["results"].items():
        row = {"run": run_name, "task": task_name}
        for metric_name, metric_value in metrics.items():
            if "stderr" in metric_name:
                continue
            if isinstance(metric_value, Number):
                row[metric_name] = float(metric_value)
            elif hasattr(metric_value, "item") and callable(metric_value.item):
                row[metric_name] = float(metric_value.item())
        rows.append(row)
    return rows


def display_lm_eval_summary(*named_results):
    rows = []
    for run_name, results in named_results:
        rows.extend(summarize_lm_eval(results, run_name))

    try:
        import pandas as pd
        return pd.DataFrame(rows)
    except Exception:
        return rows


In [ ]:
original_lm_eval_results, original_lm_eval_path = run_lm_eval_with_loaded_model(
    model,
    tokenizer,
    run_name="original_llama32_1b",
)

display_lm_eval_summary(("original", original_lm_eval_results))


In [ ]:
available_pruned_variants = []
for run_name, variable_name in (
    ("pruned_attention_group", "pruned_attention_group_model"),
    ("pruned_mlp", "pruned_mlp_model"),
    ("pruned_embedding", "pruned_embedding_model"),
):
    if variable_name in globals() and globals()[variable_name] is not None:
        available_pruned_variants.append((run_name, globals()[variable_name]))

if available_pruned_variants:
    if device == "cuda" and FREE_BASELINE_FROM_GPU_BEFORE_PRUNED_EVAL:
        model = model.to("cpu")
        torch.cuda.empty_cache()

    named_results = []
    if "original_lm_eval_results" in globals():
        named_results.append(("original", original_lm_eval_results))

    for run_name, variant_model in available_pruned_variants:
        variant_results, variant_path = run_lm_eval_with_loaded_model(
            variant_model,
            tokenizer,
            run_name=f"{run_name}_llama32_1b",
        )
        globals()[f"{run_name}_lm_eval_results"] = variant_results
        globals()[f"{run_name}_lm_eval_path"] = variant_path
        named_results.append((run_name, variant_results))

    display_lm_eval_summary(*named_results)
else:
    print("Run one or more prune cells above first if you want to evaluate pruned variants.")
